# Prophet Forecasting

Continues from [`2_forecasting_models.ipynb`](2_forecasting_models.ipynb): forecasts the same NVIDIA Close / semiconductor billings series, this time using Facebook/Meta's [Prophet](https://facebook.github.io/prophet/) library instead of classical statsmodels methods (exponential smoothing, ARIMA/SARIMA/SARIMAX).

### What Prophet actually does

Prophet is **not** an ETS model (that's what Holt-Winters in `2_` does -- separately smoothing Error/Trend/Seasonal components with recency-weighted averages). Prophet is a **decomposable additive regression model** (a GAM -- Generalized Additive Model): each component is fit as its own curve via regression, then summed:

$$y_t = g_t + s_t + h_t + \epsilon_t$$

- **$g_t$ (trend)** -- a **piecewise linear** (or logistic-growth) curve, with automatically detected **changepoints** where the growth rate is allowed to shift. Nothing like $\alpha$/$\phi$ smoothing in ETS -- it's fit by regression, not by recursively updating a smoothed level/trend from each new observation.
- **$s_t$ (seasonality)** -- modelled with a **Fourier series** (sums of sine/cosine terms) rather than the fixed per-period seasonal indices ETS uses. This lets it represent multiple seasonalities at once (e.g. weekly *and* yearly) in one term.
- **$h_t$ (holidays/events)** -- a **separate, explicit component**: you supply a list of dates (e.g. earnings releases), and Prophet fits each one's effect as its own regressor. ETS/Holt-Winters has no equivalent -- this is the piece you were thinking of.
- **$\epsilon_t$** -- leftover noise, same idea as any other model's residual.

So: holidays are handled as their own explicit term. But the rest isn't ETS-style exponential smoothing; it's closer to fitting several independent regression curves (trend, seasonality, holidays) and adding them together, which is also why Prophet doesn't require a stationary series or manual $(p,d,q)$/ACF-PACF tuning the way ARIMA does.

### Prophet vs. ARIMA/SARIMAX

Prophet does **not** use ARIMA or SARIMAX internally -- it's a separate modelling family, not a wrapper around one:

| | ARIMA / SARIMAX (`2_`) | Prophet (`3_`) |
|---|---|---|
| **Core mechanism** | regresses on **lagged values** ($p$) and **lagged forecast errors** ($q$), on a differenced/stationary series | regresses on **time itself** -- fits curves for trend, seasonality, holidays as functions of the date |
| **Stationarity** | required (differencing, order $d$) | not needed -- fits the raw, non-differenced series directly |
| **Parameter selection** | manual/ACF-PACF or AIC/BIC search over $(p,d,q)(P,D,Q,s)$ | mostly automatic (changepoint detection, Fourier term counts) |
| **What it "remembers"** | the series' own recent past (autocorrelation structure) | nothing recursive -- trend/seasonality are curves evaluated at any date, past or future |
| **Forecasting mechanism** | recursive: each future step is generated from the model's own prior forecasts/errors, one step at a time | direct: `make_future_dataframe(periods=N)` extends the date range, then a single `predict()` call evaluates the fitted curves at every one of those dates, including all future ones at once |

**Summary:**
- **ARIMA/SARIMAX = autoregressive** -- learns from the series' *own lagged values and past forecast errors*.
  - *NOTE:* Foreasting further into the future would **use prior forecasted points**. Therefore, **forecast uncertainity widens.**
  - *NOTE:* This widening is **mechanical** -- stacking $h$ shock terms recursively grows variance with $h$, even with no real change. A real future **structural break** is a separate, **unaccounted-for** risk -- ARIMA has no mechanism to anticipate one.
- **Prophet = additive curve-fitting** -- learns trend, seasonality, and holidays as *functions of the calendar date*, then sums them.
  - *NOTE:* Uncertainty widens for a different reason -- Prophet simulates **possible future trend changepoints** (it doesn't know if/when the growth rate will shift next), so the further out you forecast, the more room there is for an unseen changepoint to have occurred.
  - *NOTE:* Structural-break risk is **shared** with ARIMA -- neither sees it coming. But Prophet **explicitly simulates future changepoints** (Monte-Carlo, from historical changepoint frequency), so that risk is **partially priced in by design**; ARIMA's widening is pure **compounding of forecast variance**, with no structural-risk modelling at all.
- **Independent approaches** -- Prophet doesn't call or depend on ARIMA/SARIMAX; `2_` and `3_` forecast the *same* series (NVIDIA Close / semiconductor billings) as separate alternatives, not a chain.
- **Practical payoff** -- no stationarity/differencing prep and no manual $(p,d,q)$ tuning needed before fitting Prophet.

### How Monte Carlo builds Prophet's uncertainty interval

Prophet's trend is **piecewise linear**, with rate changes $\delta_j$ at changepoints $s_j$:

$$g(t) = \left(k + \sum_{j:\, s_j < t} \delta_j\right) t + c(t)$$

- $k$ = base growth rate, $\delta_j$ = the rate *adjustment* at changepoint $s_j$, $c(t)$ = an offset term keeping the line continuous at each changepoint.

**Fitting (historical data):** each $\delta_j \sim \text{Laplace}(0, \tau)$ -- most changepoints get a near-zero adjustment, occasionally a larger one, $\tau$ controlling flexibility. The historical **changepoint rate** is $\lambda = S / T$ ($S$ = number of changepoints, $T$ = length of history).

**Simulating the future:** to forecast $H$ steps ahead, Prophet doesn't compute one interval formula -- it draws $M$ random future trend paths (a Monte Carlo simulation) and reads the spread of outcomes directly off them:

1. For each simulation $i = 1, \dots, M$: draw a number of **new** future changepoints from $\text{Poisson}(\lambda H)$ -- the same average rate seen historically.
2. For each new changepoint, draw its rate change $\delta_{new} \sim \text{Laplace}(0, \tau)$ (same $\tau$ fitted from history).
3. Extend the trend forward with these simulated changepoints => one possible future path $g^{(i)}(t)$.

Repeating this $M$ times (Prophet defaults to $M=1000$) gives a **distribution** of plausible future trends at each date $t$. `yhat_lower`/`yhat_upper` are just the lower/upper **quantiles** of that simulated distribution (e.g. the 10th/90th percentile for an 80% interval) -- so the interval literally *is* "how much did the simulated future changepoints move the trend, across all $M$ draws," not a closed-form variance formula the way ARIMA's is.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet # Facebook/Meta's forecasting library -- fits trend + seasonality + holiday effects directly, no manual differencing/ACF-PACF tuning required
from prophet.diagnostics import cross_validation, performance_metrics # rolling-origin cross-validation and MAE/RMSE/MAPE scoring for a fitted Prophet model
from prophet.plot import plot_cross_validation_metric, plot_plotly, plot_components_plotly # plotting helpers for CV metrics and interactive forecast/component plots

final = pd.read_pickle("data/final.pkl") # prepared dataframe from 1_data_preparation.ipynb -- same source 2_forecasting_models.ipynb loads (see dependency check above: no extra data needed from 2_)
final.head()

Importing plotly failed. Interactive plots will not work.


,Date_x,Close,High,Low,Open,Volume,Close_lag1,Close_lag2,rolling_mean_3_x,rolling_sd_3_x,...,expanding_mean_y,Value_growth,Year_y,Month_y,log_Close,log_Value,boxcox_Close,diff_Close,diff_Value,seasonal_diff_Close
12,2023-04-03,27.907740,27.942668,27.280028,27.452674,398716000,27.720123,27.326929,27.651597,0.296407,...,19.645842,-0.114619,2023,4,3.328904,2.175433,6.241134,5.260063,-1.624,1.275326
13,2023-05-01,28.850805,28.998503,27.723120,27.782996,570329000,27.692184,27.170252,27.904414,0.860143,...,19.938811,-0.191382,2023,5,3.362138,2.297170,6.346521,0.943066,1.140,9.376001
14,2023-06-01,39.688572,39.967997,38.261500,38.410193,635873000,37.756531,40.028870,39.157991,1.225569,...,20.216389,0.144186,2023,6,3.681063,2.509599,7.421725,10.837767,2.354,21.423155
15,2023-07-03,42.330528,42.814586,42.119940,42.434326,198209000,42.219753,40.742630,41.764304,0.886527,...,20.442571,-0.063589,2023,7,3.745509,2.374906,7.653731,2.641956,-1.550,27.847735
16,2023-08-01,46.416576,46.808814,45.937510,46.369667,237858000,46.638149,46.659100,46.571275,0.134382,...,20.727647,-0.126332,2023,8,3.837657,2.440606,7.994558,4.086048,0.730,28.026630


### Preparing data for Prophet

Prophet requires an input dataframe with exactly two columns: **`ds`** (datestamp) and **`y`** (the value to forecast) -- unlike the DatetimeIndex-based series used for statsmodels in `2_`.

In [2]:
prophet_df = final[['Date_x', 'Close']].rename(columns={'Date_x': 'ds', 'Close': 'y'}) # Prophet's required column names
prophet_df.head()

,ds,y
12,2023-04-03,27.907740
13,2023-05-01,28.850805
14,2023-06-01,39.688572
15,2023-07-03,42.330528
16,2023-08-01,46.416576
